# Deploy RAG Agent (dev)
Este notebook **no genera** `rag_agent.py`. Asume que ya existe en:

`/Workspace/Users/<tu_usuario>/bind_agent/rag/rag_agent.py`

Objetivo:
1) Log + register del modelo en **Unity Catalog** (MLflow registry).
2) Crear/actualizar un **Model Serving endpoint** que sirva ese modelo.
3) Probar una invocación simple.

> Nota: en Compute Serverless suele faltar `mlflow`; por eso instalamos dependencias en la primera celda.


In [0]:
# # ==============================
# # 1) Dependencias (solo para pruebas locales)
# # ==============================
# En Serverless compute puede no venir mlflow instalado.
%pip install -U databricks-vectorsearch mlflow databricks-sdk
dbutils.library.restartPython()


In [0]:

# # ==============================
# # 2) Config (solo para pruebas locales, en prod se toman del asset bundle)
# # ==============================
import os, importlib.util
from pathlib import Path

def p(name, default):
    try:
        v = dbutils.widgets.get(name)
        return v if v not in (None, "") else default
    except:
        return default

BUNDLE_FILE_PATH = p("BUNDLE_FILE_PATH", "")
print(BUNDLE_FILE_PATH)

# --- Vector Search ---
VS_ENDPOINT = p("VS_ENDPOINT", "bind_agent_vs")
VS_INDEX_FULL_NAME = p("VS_INDEX_FULL_NAME", "bind_agent.docs.pdf_chunks_vs_idx")

# --- Endpoints ---
EMBED_ENDPOINT = p("EMBED_ENDPOINT", "databricks-bge-large-en")
LLM_ENDPOINT   = p("LLM_ENDPOINT", "databricks-llama-4-maverick")

# --- Retrieval params ---
TOP_K_CANDIDATES    = p("TOP_K_CANDIDATES", "120")
TOP_K_FINAL         = p("TOP_K_FINAL", "8")
LEX_FALLBACK_LIMIT  = p("LEX_FALLBACK_LIMIT", "40")

# --- Prompt/context limits ---
MAX_CONTEXT_CHARS     = p("MAX_CONTEXT_CHARS", "60000")
RERANK_SNIPPET_CHARS  = p("RERANK_SNIPPET_CHARS", "3600")

# --- LLM params ---
TEMPERATURE_RERANK  = p("TEMPERATURE_RERANK", "0.0")
TEMPERATURE_ANSWER  = p("TEMPERATURE_ANSWER", "0.2")
MAX_TOKENS_ANSWER   = p("MAX_TOKENS_ANSWER", "900")

# --- Retry params ---
MAX_RETRIES        = p("MAX_RETRIES", "3")
RETRY_SLEEP_SECS   = p("RETRY_SLEEP_SECS", "1.0")

# --- Silver SQL table  ---
RAG_SQL_TABLE = p("RAG_SQL_TABLE", "bind_agent.docs.silver_excel")  

# --- delta log table ---
RAG_DELTA_LOG_TABLE = p("RAG_DELTA_LOG_TABLE", "bind_agent.docs.rag_query_log") 

# --- Warehouse id  ---
RAG_SQL_WAREHOUSE_ID = p("RAG_SQL_WAREHOUSE_ID", "1b632a5315f277d9")  

# UC model
UC_MODEL_NAME = p("UC_MODEL_NAME", "bind_agent.docs.rag_agent")

# Experimento: usar ruta en /Users/... para evitar problemas de permisos
current_user = spark.sql("select current_user() as u").first()["u"]
EXPERIMENT_PATH = f"/Users/{current_user}/rag_agent_deploy"

# Serving endpoint
MODEL_SERVING_ENDPOINT = p("MODEL_SERVING_ENDPOINT", "bind_agent_rag_agent")

# Service Principal para el serving endpoint
SERVING_SP_NAME = p("SERVING_SP_NAME", "sp-bind-rag-agent-serving")

# Secret scope/key donde esta el client_secret del SP
# El secret se referencia con la sintaxis {{secrets/scope/key}} en env vars del endpoint
SERVING_SP_SECRET_SCOPE = p("SERVING_SP_SECRET_SCOPE", "bind-agent-secrets")
SERVING_SP_SECRET_KEY   = p("SERVING_SP_SECRET_KEY", "sp-bind-rag-agent-client-secret")

In [0]:
# ==============================
# 2.2) Setup unico: crear OAuth secret del SP y guardarlo en Databricks Secrets
# ==============================
# Ejecutar esta celda UNA SOLA VEZ. Despues se puede comentar o eliminar.
# Genera un client_secret para el SP y lo almacena en un secret scope.
#
# IMPORTANTE: el client_secret solo se muestra una vez al crearlo.
# Si se pierde, hay que generar uno nuevo.

import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host.rstrip('/')
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
_headers = {"Authorization": f"Bearer {token}"}

# --- 1) Crear secret scope (si no existe) ---
existing_scopes = [s.name for s in w.secrets.list_scopes()]
if SERVING_SP_SECRET_SCOPE in existing_scopes:
    print(f"Scope '{SERVING_SP_SECRET_SCOPE}' ya existe, OK.")
else:
    w.secrets.create_scope(scope=SERVING_SP_SECRET_SCOPE)
    print(f"✅ Scope creado: {SERVING_SP_SECRET_SCOPE}")

# --- 2) Resolver el SP ---
sp_iter = w.service_principals.list(filter=f"displayName eq '{SERVING_SP_NAME}'")
sp_obj = next(sp_iter, None)
if sp_obj is None:
    raise RuntimeError(f"SP '{SERVING_SP_NAME}' no encontrado en el workspace.")
print(f"SP: {sp_obj.display_name} (id={sp_obj.id}, app_id={sp_obj.application_id})")

# --- 3) Generar OAuth secret para el SP via REST API ---
# POST /api/2.0/accounts/.../servicePrincipals/{id}/credentials/secrets
# En workspace-level la ruta es:
resp = requests.post(
    f"{host}/api/2.0/accounts/servicePrincipals/{sp_obj.id}/credentials/secrets",
    headers=_headers,
    json={},
)

# Si la ruta de workspace no funciona, intentar la alternativa
if resp.status_code in (404, 405):
    print(f"  Ruta workspace retorno {resp.status_code}, intentando ruta alternativa...")
    resp = requests.post(
        f"{host}/api/2.0/servicePrincipals/{sp_obj.id}/credentials/secrets",
        headers=_headers,
        json={},
    )

if resp.status_code not in (200, 201):
    raise RuntimeError(
        f"No se pudo generar OAuth secret para el SP. "
        f"Status: {resp.status_code}, Response: {resp.text[:300]}\n\n"
        f"Alternativa manual:\n"
        f"  1) Generar el secret desde Account Console o Entra ID\n"
        f"  2) Guardarlo con:\n"
        f"     w.secrets.put_secret(scope='{SERVING_SP_SECRET_SCOPE}', "
        f"key='{SERVING_SP_SECRET_KEY}', string_value='<client_secret>')"
    )

secret_data = resp.json()
client_secret = secret_data.get("secret") or secret_data.get("value")
secret_id = secret_data.get("id") or secret_data.get("secret_id")

if not client_secret:
    raise RuntimeError(
        f"La API no devolvio el secret. Response keys: {list(secret_data.keys())}\n"
        f"Puede que necesites generarlo desde Account Console."
    )

print(f"✅ OAuth secret generado (secret_id={secret_id})")

# --- 4) Guardar el client_secret en Databricks Secrets ---
w.secrets.put_secret(
    scope=SERVING_SP_SECRET_SCOPE,
    key=SERVING_SP_SECRET_KEY,
    string_value=client_secret,
)
print(f"✅ Secret guardado en: {SERVING_SP_SECRET_SCOPE}/{SERVING_SP_SECRET_KEY}")

print(f"\n✅ Setup completo. El endpoint podra autenticarse como '{SERVING_SP_NAME}'.")
print("   Esta celda no necesita ejecutarse de nuevo.")


In [0]:
# # ==============================
# # 2.3) Config para rag y serving endpoint
# # ==============================

# 0) Setear env vars usadas en el RAG
os.environ["RAG_LLM_ENDPOINT"] = LLM_ENDPOINT
os.environ["RAG_EMBED_ENDPOINT"] = EMBED_ENDPOINT
os.environ["RAG_VS_ENDPOINT"] = VS_ENDPOINT
os.environ["RAG_VS_INDEX"] = VS_INDEX_FULL_NAME
os.environ["RAG_VS_INDEX_FULL_NAME"] = VS_INDEX_FULL_NAME
os.environ["RAG_TOP_K_CANDIDATES"] = TOP_K_CANDIDATES
os.environ["RAG_TOP_K_FINAL"] = TOP_K_FINAL
os.environ["RAG_LEX_FALLBACK_LIMIT"] = LEX_FALLBACK_LIMIT
os.environ["RAG_MAX_CONTEXT_CHARS"] = MAX_CONTEXT_CHARS
os.environ["RAG_RERANK_SNIPPET_CHARS"] = RERANK_SNIPPET_CHARS
os.environ["RAG_TEMPERATURE_RERANK"] = TEMPERATURE_RERANK
os.environ["RAG_TEMPERATURE_ANSWER"] = TEMPERATURE_ANSWER
os.environ["RAG_MAX_TOKENS_ANSWER"] = MAX_TOKENS_ANSWER
os.environ["RAG_MAX_RETRIES"] = MAX_RETRIES
os.environ["RAG_RETRY_SLEEP_SECS"] = RETRY_SLEEP_SECS
os.environ["RAG_DELTA_LOG_TABLE"] = RAG_DELTA_LOG_TABLE
os.environ["RAG_SQL_TABLE"] = RAG_SQL_TABLE
os.environ["RAG_SQL_WAREHOUSE_ID"] = RAG_SQL_WAREHOUSE_ID
#temporalmente solo para debugg
os.environ["RAG_DEBUG_TRACE_SQL"] = "1"

# env usadas en el serving endpoint
env_vars = {
    "RAG_LLM_ENDPOINT": os.environ["RAG_LLM_ENDPOINT"],
    "RAG_EMBED_ENDPOINT": os.environ["RAG_EMBED_ENDPOINT"],
    "RAG_VS_ENDPOINT": os.environ["RAG_VS_ENDPOINT"],
    "RAG_VS_INDEX": os.environ["RAG_VS_INDEX"],
    "RAG_VS_INDEX_FULL_NAME": os.environ["RAG_VS_INDEX_FULL_NAME"],
    "RAG_TOP_K_CANDIDATES": os.environ["RAG_TOP_K_CANDIDATES"],
    "RAG_TOP_K_FINAL": os.environ["RAG_TOP_K_FINAL"],
    "RAG_LEX_FALLBACK_LIMIT": os.environ["RAG_LEX_FALLBACK_LIMIT"],
    "RAG_MAX_CONTEXT_CHARS": os.environ["RAG_MAX_CONTEXT_CHARS"],
    "RAG_RERANK_SNIPPET_CHARS": os.environ["RAG_RERANK_SNIPPET_CHARS"],
    "RAG_TEMPERATURE_RERANK": os.environ["RAG_TEMPERATURE_RERANK"],
    "RAG_TEMPERATURE_ANSWER": os.environ["RAG_TEMPERATURE_ANSWER"],
    "RAG_MAX_TOKENS_ANSWER": os.environ["RAG_MAX_TOKENS_ANSWER"],
    "RAG_MAX_RETRIES": os.environ["RAG_MAX_RETRIES"],
    "RAG_RETRY_SLEEP_SECS": os.environ["RAG_RETRY_SLEEP_SECS"],
    "RAG_DELTA_LOG_TABLE": os.environ["RAG_DELTA_LOG_TABLE"],
    "RAG_SQL_TABLE": os.environ["RAG_SQL_TABLE"],
    "RAG_SQL_WAREHOUSE_ID": os.environ["RAG_SQL_WAREHOUSE_ID"],
    #temporalmente solo para debugg
    "RAG_DEBUG_TRACE_SQL": os.environ["RAG_DEBUG_TRACE_SQL"],
    # --- Identidad del SP: fuerza al endpoint a autenticarse como el SP asignado ---
    # DATABRICKS_AUTH_TYPE fuerza OAuth M2M, evitando que el SDK use model-serving.
    # DATABRICKS_HOST es necesario para que OAuth M2M resuelva el token endpoint.
    # DATABRICKS_CLIENT_SECRET usa la sintaxis {{secrets/...}} que Databricks resuelve
    # en runtime sin exponer el valor en texto plano.
    "DATABRICKS_HOST": "",              # se completa en celda 6 despues de resolver el SP
    "DATABRICKS_AUTH_TYPE": "oauth-m2m",
    "DATABRICKS_CLIENT_ID": "",         # se completa en celda 6 despues de resolver el SP
    "DATABRICKS_CLIENT_SECRET": "",     # se completa en celda 6 despues de resolver el SP
}

In [0]:
# ==============================
# 3) Localizar el paquete bind_rag_agent
# ==============================

if BUNDLE_FILE_PATH:
    project_root = Path(BUNDLE_FILE_PATH).parent
else:
    project_root = Path(f"/Workspace/Users/{current_user}/bind_agent")

bind_rag_pkg = project_root / "src" / "bind_rag_agent"
assert bind_rag_pkg.exists(), f"No se encontro bind_rag_agent en {bind_rag_pkg}"

# Verificar que tiene __init__.py
assert (bind_rag_pkg / "__init__.py").exists(), f"Falta __init__.py en {bind_rag_pkg}"

print(f"Package encontrado: {bind_rag_pkg}")
print(f"Archivos: {[f.name for f in bind_rag_pkg.iterdir() if f.suffix == '.py']}")


In [0]:
# ==============================
# 4) Log + Register (UC) - ChatModel (streaming nativo)
# ==============================
import sys
import time
import re
import mlflow
from mlflow.tracking import MlflowClient
from databricks.sdk import WorkspaceClient
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksVectorSearchIndex
from mlflow.pyfunc import ChatModel
from mlflow.types.llm import (
    ChatCompletionResponse,
    ChatCompletionChunk,
    ChatMessage,
    ChatChoice,
    ChatParams,
    ChatChoiceDelta,
    ChatChunkChoice,
)
from typing import List, Generator, Optional

resources = [
    DatabricksServingEndpoint(endpoint_name=os.environ["RAG_LLM_ENDPOINT"]),
    DatabricksServingEndpoint(endpoint_name=os.environ["RAG_EMBED_ENDPOINT"]),
    DatabricksVectorSearchIndex(index_name=os.environ["RAG_VS_INDEX_FULL_NAME"]),
]

w = WorkspaceClient()
w.workspace.mkdirs(f"/Users/{current_user}/bind_agent")

mlflow.set_experiment(EXPERIMENT_PATH)
mlflow.set_registry_uri("databricks-uc")

# Agregar el parent de bind_rag_agent al sys.path para que
# MLflow pueda validar el modelo durante log_model
pkg_parent = str(bind_rag_pkg.parent)
if pkg_parent not in sys.path:
    sys.path.insert(0, pkg_parent)


class RagChatModel(ChatModel):
    """
    ChatModel habilita streaming nativo en serving endpoints.
    """

    def load_context(self, context):
        from bind_rag_agent.rag_agent import RagAgent
        self._agent = RagAgent()

    def _run_rag(self, query: str) -> str:
        import pandas as _pd
        import json as _json
        if not hasattr(self, '_agent'):
            self.load_context(None)
        try:
            df = _pd.DataFrame([{"query": query}])
            out = self._agent.predict(None, df)
            if isinstance(out, list) and out:
                answer = out[0].get("answer", "")
                error = out[0].get("error", "")
                if error:
                    return f"Error: {error}"
                if not answer:
                    return "No se encontro informacion relevante."

                # Agregar fuentes al final de la respuesta
                sources_raw = out[0].get("sources_json", "[]")
                try:
                    sources = _json.loads(sources_raw) if isinstance(sources_raw, str) else sources_raw
                except Exception:
                    sources = []

                if sources:
                    answer += "\n\n📎 **Fuentes:**\n"
                    for s in sources:
                        parts = []
                        if s.get("path"):
                            # Solo el nombre del archivo
                            fname = s["path"].rsplit("/", 1)[-1]
                            parts.append(fname)
                        if s.get("page_num") is not None:
                            parts.append(f"p.{s['page_num']}")
                        if s.get("topic"):
                            parts.append(s["topic"])
                        answer += f"- [{s.get('sid', '?')}] {' | '.join(parts)}\n"

                return answer
            return "Sin respuesta del RAG."
        except Exception as e:
            return f"Error procesando la consulta: {e}"

    def predict(
        self, context, messages: List[ChatMessage], params: ChatParams
    ) -> ChatCompletionResponse:
        query = messages[-1].content if messages else ""
        content = self._run_rag(query) if query else "No se recibio ninguna pregunta."
        return ChatCompletionResponse(
            choices=[
                ChatChoice(
                    message=ChatMessage(role="assistant", content=content)
                )
            ]
        )

    def _create_chunk(self, content: str, finish_reason: Optional[str] = None) -> ChatCompletionChunk:
        return ChatCompletionChunk(
            choices=[
                ChatChunkChoice(
                    delta=ChatChoiceDelta(role="assistant", content=content),
                    finish_reason=finish_reason,
                )
            ]
        )

    def predict_stream(
        self, context, messages: List[ChatMessage], params: ChatParams
    ) -> Generator[ChatCompletionChunk, None, None]:
        query = messages[-1].content if messages else ""
        content = self._run_rag(query) if query else "No se recibio ninguna pregunta."

        chunk_size = 100
        for i in range(0, len(content), chunk_size):
            yield self._create_chunk(content[i:i+chunk_size])

        yield self._create_chunk("", finish_reason="stop")


with mlflow.start_run(run_name="rag_agent_deploy") as run:
    logged = mlflow.pyfunc.log_model(
        artifact_path="rag_agent",
        python_model=RagChatModel(),
        code_paths=[str(bind_rag_pkg)],
        pip_requirements=[
            "databricks-vectorsearch",
            "mlflow",
            "requests",
        ],
        resources=resources,
    )
    model_uri = logged.model_uri
    print("Model logged:", model_uri)

    uc_model_info = mlflow.register_model(model_uri=model_uri, name=UC_MODEL_NAME)
    print("Registered in UC:", uc_model_info.name, "version:", uc_model_info.version)

# Esperar READY
client = MlflowClient()
t0 = time.time()
while True:
    mv = client.get_model_version(name=UC_MODEL_NAME, version=uc_model_info.version)
    if mv.status == "READY":
        break
    if time.time() - t0 > 900:
        raise TimeoutError(f"Timeout esperando READY para {UC_MODEL_NAME} v{uc_model_info.version}")
    print("Esperando model READY... status=", mv.status)
    time.sleep(5)

print(f"Model version READY: {UC_MODEL_NAME} v{uc_model_info.version}")

# Verificar task
model_info = mlflow.models.get_model_info(model_uri)
print(f"Task metadata: {model_info.metadata}")


In [0]:
# ==============================
# 5) Crear/Actualizar Model Serving Endpoint (con SP via OAuth credentials)
# ==============================
# Estrategia: inyectar DATABRICKS_CLIENT_ID y DATABRICKS_CLIENT_SECRET como
# environment variables del served entity. Esto fuerza a WorkspaceClient()
# dentro del agente a autenticarse como el SP asignado, en vez del SP
# auto-generado por Databricks.
#
# Prerequisito: el client_secret del SP debe estar almacenado en:
#   Databricks Secrets -> scope: {SERVING_SP_SECRET_SCOPE} / key: {SERVING_SP_SECRET_KEY}
#
# Para crearlo (una sola vez):
#   databricks secrets create-scope <scope>
#   databricks secrets put-secret <scope> <key> --string-value "<client_secret>"

from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput
import datetime as dt
from databricks.sdk.errors import NotFound

# --- 1) Resolver el SP por nombre ---
print(f"Buscando SP '{SERVING_SP_NAME}' en el workspace...")
sp_iter = w.service_principals.list(filter=f"displayName eq '{SERVING_SP_NAME}'")
sp_obj = next(sp_iter, None)
if sp_obj is None:
    raise RuntimeError(
        f"No se encontro el Service Principal '{SERVING_SP_NAME}'. "
        "Verificar que exista en el workspace."
    )

SERVING_SP_ID = sp_obj.application_id   # UUID del SP (= client_id OAuth)
SERVING_SP_NUMERIC_ID = sp_obj.id        # ID numerico
print(f"  SP encontrado: {sp_obj.display_name}")
print(f"    Application ID (client_id): {SERVING_SP_ID}")
print(f"    Numeric ID               : {SERVING_SP_NUMERIC_ID}")

# --- 2) Verificar que el secret existe ---
secret_ref = f"{{{{secrets/{SERVING_SP_SECRET_SCOPE}/{SERVING_SP_SECRET_KEY}}}}}"
print(f"\nVerificando secret scope '{SERVING_SP_SECRET_SCOPE}'...")
try:
    scopes = [s.name for s in w.secrets.list_scopes()]
    assert SERVING_SP_SECRET_SCOPE in scopes, (
        f"Scope '{SERVING_SP_SECRET_SCOPE}' no existe. "
        f"Crearlo con: databricks secrets create-scope {SERVING_SP_SECRET_SCOPE}"
    )
    keys = [k.key for k in w.secrets.list_secrets(scope=SERVING_SP_SECRET_SCOPE)]
    assert SERVING_SP_SECRET_KEY in keys, (
        f"Key '{SERVING_SP_SECRET_KEY}' no existe en scope '{SERVING_SP_SECRET_SCOPE}'. "
        f"Crearlo con: databricks secrets put-secret {SERVING_SP_SECRET_SCOPE} {SERVING_SP_SECRET_KEY}"
    )
    print(f"  ✅ Secret encontrado: {SERVING_SP_SECRET_SCOPE}/{SERVING_SP_SECRET_KEY}")
except AssertionError as e:
    raise RuntimeError(str(e))
except Exception as e:
    print(f"  ⚠️ No se pudo verificar el secret: {e}")
    print(f"  Continuando asumiendo que existe...")

# --- 3) Inyectar credenciales del SP en env_vars ---
env_vars["DATABRICKS_HOST"] = w.config.host.rstrip("/")
env_vars["DATABRICKS_CLIENT_ID"] = SERVING_SP_ID
env_vars["DATABRICKS_CLIENT_SECRET"] = secret_ref
print(f"\n  env_vars actualizadas:")
print(f"    DATABRICKS_HOST          = {env_vars['DATABRICKS_HOST']}")
print(f"    DATABRICKS_AUTH_TYPE     = oauth-m2m")
print(f"    DATABRICKS_CLIENT_ID     = {SERVING_SP_ID}")
print(f"    DATABRICKS_CLIENT_SECRET = {secret_ref}")

# --- 4) Crear/actualizar endpoint ---
def ensure_serving_endpoint(endpoint_name, model_name, model_version, workload_size="Small"):
    served = ServedEntityInput(
        name=f"{endpoint_name}-entity",
        entity_name=model_name,
        entity_version=str(model_version),
        workload_size=workload_size,
        scale_to_zero_enabled=True,
        environment_vars=env_vars,
    )

    cfg = EndpointCoreConfigInput(name=endpoint_name, served_entities=[served])

    try:
        w.serving_endpoints.get(endpoint_name)
        print(f"\n[serving] Endpoint existe: {endpoint_name}")
        w.serving_endpoints.update_config_and_wait(
            name=endpoint_name,
            served_entities=[served],
            timeout=dt.timedelta(minutes=30),
        )
    except NotFound:
        print(f"\n[serving] Creando endpoint: {endpoint_name}")
        w.serving_endpoints.create_and_wait(
            name=endpoint_name,
            config=cfg,
            timeout=dt.timedelta(minutes=30),
        )

    print(f"✅ Serving endpoint listo: {endpoint_name}")
    print(f"   Identidad de ejecucion: {SERVING_SP_NAME} ({SERVING_SP_ID})")

ensure_serving_endpoint(
    endpoint_name=MODEL_SERVING_ENDPOINT,
    model_name=UC_MODEL_NAME,
    model_version=uc_model_info.version,
    workload_size="Small",
)


In [0]:
# ==============================
# 5.1) Verificar Service Principal del Serving Endpoint
# ==============================
# El SP se resolvio en la celda anterior y sus credenciales OAuth
# fueron inyectadas como env vars del endpoint.
#
# Cuando el agente instancia WorkspaceClient(), detecta automaticamente
# DATABRICKS_CLIENT_ID + DATABRICKS_CLIENT_SECRET y se autentica
# como el SP asignado (en vez del SP auto-generado).
#
# Para verificar que funciona, revisar los logs del endpoint despues
# del smoke test. Deberia aparecer:
#   [sql_evidence] Identity: {SERVING_SP_ID}

print(f"SP asignado al endpoint: {SERVING_SP_NAME}")
print(f"  Application ID (client_id): {SERVING_SP_ID}")
print(f"  Numeric ID                : {SERVING_SP_NUMERIC_ID}")
print(f"  Secret ref                : {{{{secrets/{SERVING_SP_SECRET_SCOPE}/{SERVING_SP_SECRET_KEY}}}}}")


In [0]:
# ==============================
# 5.2) Otorgar permisos al SP del Serving Endpoint
# ==============================
# El SP (resuelto por nombre en celda 5) necesita:
#   - CAN_USE sobre el SQL Warehouse (para Statement Execution API)
#   - USE CATALOG / USE SCHEMA / SELECT sobre la tabla SQL

import requests

host = w.config.host.rstrip('/')
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {"Authorization": f"Bearer {token}"}

# --- 1) Permiso CAN_USE sobre SQL Warehouse ---
print(f"Otorgando CAN_USE sobre warehouse {RAG_SQL_WAREHOUSE_ID} al SP {SERVING_SP_ID}...")

warehouse_perm_ok = False
for field in ["service_principal_name", "user_name"]:
    resp = requests.patch(
        f"{host}/api/2.0/permissions/sql/warehouses/{RAG_SQL_WAREHOUSE_ID}",
        headers=headers,
        json={
            "access_control_list": [
                {
                    field: SERVING_SP_ID,
                    "permission_level": "CAN_USE"
                }
            ]
        }
    )
    if resp.status_code == 200:
        print(f"  ✅ Warehouse: permiso otorgado (via {field})")
        warehouse_perm_ok = True
        break
    else:
        print(f"  ⚠️ {field}: {resp.status_code} - {resp.text[:200]}")

if not warehouse_perm_ok:
    print("  ❌ No se pudo otorgar permiso en el warehouse. Revisar manualmente.")

# --- 2) Permisos Unity Catalog (CATALOG / SCHEMA / TABLE) ---
print(f"\nOtorgando permisos Unity Catalog al SP {SERVING_SP_ID}...")

grants = [
    f"GRANT USE CATALOG ON CATALOG bind_agent TO `{SERVING_SP_ID}`",
    f"GRANT USE SCHEMA ON SCHEMA bind_agent.docs TO `{SERVING_SP_ID}`",
    f"GRANT SELECT ON TABLE {RAG_SQL_TABLE} TO `{SERVING_SP_ID}`",
]

for grant_sql in grants:
    try:
        spark.sql(grant_sql)
        short = grant_sql.split('TO')[0].strip()
        print(f"  ✅ {short}")
    except Exception as e:
        print(f"  ❌ {grant_sql[:80]}... → {e}")

print(f"\n✅ Permisos configurados para SP: {SERVING_SP_ID}")


In [0]:
# ==============================
# 6) Smoke test (invocar endpoint) - formato chat
# ==============================
from mlflow.deployments import get_deploy_client
dc = get_deploy_client("databricks")

payload = {
    "messages": [
        # {"role": "user", "content": "Cuales son las Previsiones de octubre 2025?"}
        # {"role": "user", "content": "Dame el resultado neto del cliente grimoldi por producto para julio 2025"}
        {"role": "user", "content": "Dame el resultado neto del cliente santander por producto para julio 2025"}
    ]
}

resp = dc.predict(endpoint=MODEL_SERVING_ENDPOINT, inputs=payload)
# print(resp)


In [0]:
# from databricks.sdk import WorkspaceClient
# w = WorkspaceClient()
# logs = w.serving_endpoints.logs(
#     name="bind_agent_rag_agent",
#     served_model_name="bind_agent_rag_agent-entity"
# )
# print(logs.logs)

In [0]:
# ==============================
# 7) Nada que limpiar (sin directorio temporal)
# ==============================
print("Deploy completo.")
